# Required Imports

In [1]:
import os
import pandas as pd
import re
import matplotlib.pyplot as plt
from io import BytesIO
from scipy.io import loadmat  # this is the SciPy module that loads mat-files
import numpy as np

import torch
from torch_geometric.data import Data

# Imports PIL module
from PIL import Image

In [2]:
# Configuration Settings
n_packets = 200 # Espandiamo la finestra a 200 pacchetti (circa 2 secondi di camminata)
stride = 50     # La finestra avanza di 50 pacchetti alla volta (75% di overlap)

# Dataset Generation

## Load IQR Amplitude values 

In [3]:
import os
import pandas as pd
import re

def extract_id_from_filename(filename):
    match = re.search(r'(\d+[mf][AB])\.', filename)
    return match.group(1) if match else None

def load_feature_dicts(directory):
    feature_dict = {}
    for file_name in os.listdir(directory):
        if file_name.endswith('.csv'):
            file_path = os.path.join(directory, file_name)
            df = pd.read_csv(file_path)
            file_id = extract_id_from_filename(file_name)
            if file_id:
                feature_dict[file_id] = df
    return feature_dict

output_folder = './output/'
amp_input_directory = os.path.join(output_folder, 'CSV', 'sanitized_amplitudes', 'IQR_amplitudes')
phase_input_directory = os.path.join(output_folder, 'CSV', 'raw_phases')

sanitized_amplitudes = load_feature_dicts(amp_input_directory)
sanitized_phases = load_feature_dicts(phase_input_directory)

In [4]:
split_sanitized_data = {}

for ID in sanitized_amplitudes.keys():
    split_sanitized_data[ID] = []
    amp_values = sanitized_amplitudes[ID]
    phase_values = sanitized_phases[ID] 
    
    total_len = len(amp_values)

    # Scorriamo la finestra usando il passo definito da 'stride'
    for start in range(0, total_len - n_packets + 1, stride):
        end = start + n_packets
        
        # Estraiamo ampiezza e fase, rimuoviamo il timestamp e convertiamo in numpy
        amp_group = amp_values.iloc[start:end].drop(columns='timestamp').to_numpy()
        phase_group = phase_values.iloc[start:end].drop(columns='timestamp').to_numpy()
        
        # Concateniamo sull'asse delle feature
        # Se n_packets=200, la shape finale sarà [52 nodi, 400 feature]
        combined_features = np.concatenate((amp_group.T, phase_group.T), axis=1)
        
        split_sanitized_data[ID].append(combined_features)

## Split data into Train, Validation, and Test sets

In [5]:
import random

# Set the seed to make the split reproducible across experiments
random.seed(42)

# Group packets by base identity (e.g., from '00001mA' to '00001m')
groups = {}
for key, packets in split_sanitized_data.items():
    base_id = key[:-1] # Get everything except the last letter (which is 'A' or 'B')
    session = key[-1]  # 'A' or 'B'
    
    if base_id not in groups:
        groups[base_id] = {'A': [], 'B': []}
    
    # Add the packets to their respective session
    groups[base_id][session].extend(packets)

# Extract all 81 unique identities and shuffle them
all_identities = list(groups.keys())
random.shuffle(all_identities)

# Re-ID split by IDENTITIES: 40 IDs for Training, 41 IDs for Test
train_ids = all_identities[:40]
test_ids = all_identities[40:]

print(f"Subjects in Training Set: {len(train_ids)}")
print(f"Subjects in Test Set: {len(test_ids)}")

Subjects in Training Set: 40
Subjects in Test Set: 41


## Generate Probes and Gallery for each set

In [6]:
train_probe = {}
train_gallery = {}
test_probe = {}
test_gallery = {}
val_probe = {}   # Initialized empty because in a pure Re-ID setup, validation 
val_gallery = {} # is either done via cross-fold on the train set, or the original set isn't used.

# 1. Training Set Construction (Merge Session A and Session B)
for base_id in train_ids:
    # Combine heatmaps from Session A and Session B to enforce invariance
    all_packets = groups[base_id]['A'] + groups[base_id]['B']
    
    # Split the total packets in half (half for probe, half for gallery) 
    # so the Siamese Network has pairs to learn the loss
    half = len(all_packets) // 2
    train_probe[base_id] = all_packets[:half]
    train_gallery[base_id] = all_packets[half:]

# 2. Test Set Construction (Gallery = Session A, Probe = Session B)
for base_id in test_ids:
    # Gallery: database of known identities (Session A only)
    test_gallery[base_id] = groups[base_id]['A']
    
    # Probe: queries to search for (Session B only)
    test_probe[base_id] = groups[base_id]['B']
    
print(f"Generated training data for {len(train_probe)} identities.")
print(f"Generated test data (Gallery A / Probe B) for {len(test_probe)} identities.")

Generated training data for 40 identities.
Generated test data (Gallery A / Probe B) for 41 identities.


## Generate Magnitude Heatmaps



In [7]:
import torch
from torch_geometric.data import Data
import os

def save_graph_data(data_matrix, subject_id, output_path):
    # Nodes (x) are already [52, 200]
    x = torch.tensor(data_matrix, dtype=torch.float32)
    
    # Fully connected edge generation remains unchanged
    num_nodes = x.shape[0] 
    row_idx, col_idx = torch.meshgrid(torch.arange(num_nodes), torch.arange(num_nodes), indexing='ij')
    edge_index = torch.stack([row_idx.flatten(), col_idx.flatten()], dim=0).to(torch.long)
    
    graph_data = Data(x=x, edge_index=edge_index, subject_id=subject_id)
    torch.save(graph_data, output_path)

##################### TRAIN GRAPHS ##########################

# Generate and save graph data for train probes
if train_probe:
  # Folder to save the train probes
  output_path = os.path.join(output_folder, 'GraphData', 'D1', 'Probe')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the train_probe dictionary
  for ID, group_list in train_probe.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_matrix in enumerate(group_list):
          filename = f"{ID}_{group_idx}.pt"
          # Pass the matrix directly since it's already a cleaned NumPy array
          save_graph_data(group_matrix, ID, os.path.join(output_path, filename))

# Generate and save graph data for train gallery
if train_gallery:
  # Folder to save the train gallery
  output_path = os.path.join(output_folder, 'GraphData', 'D1', 'Gallery')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the train_gallery dictionary
  for ID, group_list in train_gallery.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_matrix in enumerate(group_list):
          filename = f"{ID}_{group_idx}.pt"
          # Pass the matrix directly since it's already a cleaned NumPy array
          save_graph_data(group_matrix, ID, os.path.join(output_path, filename))

##################### VAL GRAPHS ############################

# Generate and save graph data for val probes
if val_probe:
  # Folder to save the val probes
  output_path = os.path.join(output_folder, 'GraphData', 'D2v', 'Probe')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the val_probe dictionary
  for ID, group_list in val_probe.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_matrix in enumerate(group_list):
          filename = f"{ID}_{group_idx}.pt"
          # Pass the matrix directly since it's already a cleaned NumPy array
          save_graph_data(group_matrix, ID, os.path.join(output_path, filename))

# Generate and save graph data for val gallery
if val_gallery:
  # Folder to save the val gallery
  output_path = os.path.join(output_folder, 'GraphData', 'D2v', 'Gallery')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the val_gallery dictionary
  for ID, group_list in val_gallery.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_matrix in enumerate(group_list):
          filename = f"{ID}_{group_idx}.pt"
          # Pass the matrix directly since it's already a cleaned NumPy array
          save_graph_data(group_matrix, ID, os.path.join(output_path, filename))

##################### TEST GRAPHS ##########################

# Generate and save graph data for test probes
if test_probe:
  # Folder to save the test probes
  output_path = os.path.join(output_folder, 'GraphData', 'D2t', 'Probe')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the test_probe dictionary
  for ID, group_list in test_probe.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_matrix in enumerate(group_list):
          filename = f"{ID}_{group_idx}.pt"
          # Pass the matrix directly since it's already a cleaned NumPy array
          save_graph_data(group_matrix, ID, os.path.join(output_path, filename))

# Generate and save graph data for test gallery
if test_gallery:
  # Folder to save the test gallery
  output_path = os.path.join(output_folder, 'GraphData', 'D2t', 'Gallery')
  if not os.path.exists(output_path):
    os.makedirs(output_path)

  # Iterate over each ID in the test_gallery dictionary
  for ID, group_list in test_gallery.items():
      # Iterate over each packet group for the current ID
      for group_idx, group_matrix in enumerate(group_list):
          filename = f"{ID}_{group_idx}.pt"
          # Pass the matrix directly since it's already a cleaned NumPy array
          save_graph_data(group_matrix, ID, os.path.join(output_path, filename))

#!zip -r /content/CSV.zip /content/output/CSV
#!zip -r /content/GraphData.zip /content/output/GraphData

In [8]:
import torch
from torch_geometric.data import Data
import os

# Definiamo la cartella invece del singolo file
probe_dir = os.path.join(output_folder, 'GraphData', 'D1', 'Probe')

if os.path.exists(probe_dir):
    # Peschiamo tutti i file .pt nella cartella
    pt_files = [f for f in os.listdir(probe_dir) if f.endswith('.pt')]
    
    if pt_files:
        # Prendiamo il primissimo file della lista
        sample_file = os.path.join(probe_dir, pt_files[0])
        print(f"File trovato con successo: {sample_file}\n")
        
        # 1. Carica il grafo
        graph_data = torch.load(sample_file, weights_only=False)
        
        # 2. Print general overview
        print("=== Object Overview ===")
        print(graph_data)
        
        # 3. Check exact dimensions (Sanity Check)
        print("\n=== Tensor Details ===")
        print(f"Node shape (x): {graph_data.x.shape}") 
        print(f"Edge shape (edge_index): {graph_data.edge_index.shape}")
        print(f"Saved Subject ID: {graph_data.subject_id}")
        
        # 4. Useful PyTorch Geometric native methods
        print("\n=== Structural Properties ===")
        print(f"Total number of nodes: {graph_data.num_nodes}")
        print(f"Total number of edges: {graph_data.num_edges}")
        print(f"Are there isolated nodes? {graph_data.has_isolated_nodes()}")
    else:
        print("La cartella esiste, ma è vuota. Nessun file .pt trovato al suo interno.")
else:
    print(f"Cartella non trovata: {probe_dir}. Assicurati che lo script di generazione abbia girato correttamente.")

File trovato con successo: ./output/GraphData/D1/Probe/00038m_2.pt

=== Object Overview ===
Data(x=[52, 400], edge_index=[2, 2704], subject_id='00038m')

=== Tensor Details ===
Node shape (x): torch.Size([52, 400])
Edge shape (edge_index): torch.Size([2, 2704])
Saved Subject ID: 00038m

=== Structural Properties ===
Total number of nodes: 52
Total number of edges: 2704
Are there isolated nodes? False
